<a id="top"></a>

# Meet LangChain & LangGraph: the framework the graph lessons use

```yaml
title: "Meet LangChain & LangGraph: the framework the graph lessons use"
keywords: langchain, langgraph, ChatAnthropic, chat model, messages, ainvoke, AIMessage, framework, client abstraction, LangChain vs LangGraph, primer
estimated duration: 12
```

> **Lesson:** L03 — Directed graphs: from one node to a sequential chain (**Movement 1**). A short
> **primer** to read *before* the build demo
> ([L0303_lecture.ipynb](L0303_lecture.ipynb)): it isolates *what a framework is* and the few
> **LangChain** basics the build demo assumes, so the demo can focus on the *node*.
> Makes a couple of **small live** `ChatAnthropic` calls — set `ANTHROPIC_API_KEY` to run them
> (a keyless restart-and-run-all still passes; the live cells are gated on `LIVE`).
> **Anchor model: Claude Sonnet 4.6.** Clear outputs before committing.

## Contents

- [1. Why a framework now?](#1-why-a-framework-now)
- [2. `ChatAnthropic`: a chat model you call](#2-chatanthropic-a-chat-model-you-call)
- [3. Messages in, message out](#3-messages-in-message-out)
- [4. LangChain vs LangGraph](#4-langchain-vs-langgraph)
- [5. What we're not doing yet](#5-what-were-not-doing-yet)

## 1. Why a framework now?

In [L01](../../../../docs/origin/lesson_roadmaps/L01/objectives.md) and
[L02](../../../../docs/origin/lesson_roadmaps/L02/objectives.md) you talked to the model through the
repo's own hand-rolled [`potato_llm`](../../potato_llm/CLAUDE.md) seam. From L03 on, the course
**reaches for a framework** — and a framework brings its own **client abstraction**.

A *framework* is pre-built scaffolding for a common job so you don't re-invent it. Here that job is
two separable things you'll meet in a moment:

- **LangChain** — a standard way to *talk to a model*: one `ChatAnthropic` object you call, with the
  system/user roles from L02 turned into small message objects.
- **LangGraph** — a standard way to *wire steps together*: state, nodes, and edges (the build demo).

Underneath, it is still **just an API call** — same tokens, same cost, same key (still loaded
through [`common/config.py`](../../common/config.py)). Only the *client* is the framework's now. This
primer meets LangChain first; the build demo then wires one LangChain call into a LangGraph node.

[↑ Back to top](#top)

## 2. `ChatAnthropic`: a chat model you call

**LangChain's `ChatAnthropic` is a chat-model client**: you build one, then `await` its
`.ainvoke(...)` to get a reply. That is the whole surface for now. Build it once (below) — a single
Sonnet client, kept small so a class run costs cents — then call it.

> We call `chat.ainvoke(...)` — the **async** (awaitable) twin of `chat.invoke(...)`. Every
> LangChain runnable offers both; the course defaults to the async one, so you `await` model calls
> (a notebook cell can `await` at top level). *Why* async is the default is the K05 prework's
> "why async for agents."

The client is built only when a key is present, so a keyless restart-and-run-all still passes; the
live call cells are gated on `LIVE`. This is the first code cell — run it first.

In [ ]:
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage, SystemMessage

from fluffy_potato_curriculum.common.config import get_settings, require_anthropic_key

# The course anchor model. Movement 1 keeps the model CONSTANT -- Sonnet only -- so the *framework*
# is the only new thing to track. (Movement 2 mixes models per node.)
SONNET = "claude-sonnet-4-6"

# Live calls load the key through the config seam (common/config.py) -- same key as L01-L02, only the
# *client* is the framework's now. Build the client only when a key is present so a keyless
# restart-and-run-all still passes; the call cells below are gated on LIVE.
LIVE = bool(get_settings().anthropic_api_key)
if LIVE:
    # One small client (low max_tokens) so a class run costs cents, not dollars.
    chat = ChatAnthropic(model=SONNET, api_key=require_anthropic_key(), max_tokens=256)
else:
    print("No ANTHROPIC_API_KEY set -- concept cells run; the live call cells will skip.")

In [ ]:
# The simplest possible call: a plain string in, a reply object out. Read the text off `.content`.
if not LIVE:
    print("No ANTHROPIC_API_KEY set -- skipping the live call.")
else:
    reply = await chat.ainvoke("In one sentence, what is LangChain?")
    print(reply.content)

[↑ Back to top](#top)

## 3. Messages in, message out

A plain string is the shortcut. The real input is a **list of messages** — and those messages are
just the **system/user roles from L02**, now as small objects: a `SystemMessage` (the standing
instruction) and a `HumanMessage` (the turn). `await chat.ainvoke(...)` returns one **`AIMessage`**,
and the text is on its `.content`.

So the L02 idea — *roles shape the reply* — is unchanged. LangChain only gives it a typed shape:
messages in, one message out.

In [ ]:
if not LIVE:
    print("No ANTHROPIC_API_KEY set -- skipping the live call.")
else:
    messages = [
        SystemMessage(content="You are a terse assistant. Answer in one short sentence."),
        HumanMessage(content="What is the difference between LangChain and LangGraph?"),
    ]
    reply = await chat.ainvoke(messages)
    print("return type:", type(reply).__name__)  # AIMessage
    print("content    :", reply.content)

[↑ Back to top](#top)

## 4. LangChain vs LangGraph

One line to hold onto: **LangChain talks to the model; LangGraph wires steps.** They are separate
libraries you use together — you just used LangChain (`ChatAnthropic`, messages); the build demo uses
LangGraph to wrap that call in a node.

| | LangChain | LangGraph |
| --- | --- | --- |
| Job | talk to the model | wire steps together |
| You meet it | here (`ChatAnthropic`, messages, `.ainvoke()`) | the build demo (`StateGraph`, nodes, edges) |
| Unit | one model call | a graph of nodes |
| Analogy | the phone call | the org chart of who calls whom |

A LangGraph **node** is just one LangChain call — like the ones above — wrapped so an orchestrator
can plug it in. That is the entire bridge into the build demo: you already wrote the call; the demo
only *wires* it.

[↑ Back to top](#top)

## 5. What we're not doing yet

Scope, so the new words don't sprawl. Today is **only** "what is a framework, and how do I call
LangChain's model client." Deliberately **not** here:

- **`StateGraph` internals** — typed state, nodes, `compile`/`ainvoke`. That is the very next item,
  the build demo ([L0303_lecture.ipynb](L0303_lecture.ipynb)).
- **Tools** — letting the model call your functions. That is **L07**.
- **An agent loop / "harness"** — the model driving its own control flow in a loop. The graphs in
  L03 & L05 are **workflows** (*you* wire the path); the model-driven **agent loop** (often called a
  *harness*) is deliberately reserved for **L10–L11**. Keep the two words apart: a workflow is wired
  by you; a harness lets the model loop.

> **Next up:** the build demo wires one of these LangChain calls into your first LangGraph node.

[↑ Back to top](#top)